# Connecting Causality and Deep Learning
Notes and opinionated code

## Causal Abstraction and Plate Models

After framing up the subject at hand, Robert begins with digit identifying use case and highlights the futility of trying to determine whether one pixel causes another (5.4 the unweildy naive DAG). This indicates we need a higher level of abstraction. 

Necessary to adapt because unlike challenges with income, social data, etc., in deep learning we're at infintessimile levels. We wouldn't reason on atoms to understand the price of a burger. We wouldn't reason on microsounds to understand speech

The author presents "Plate Modeling" as a way to define abstractions. If the plate is "Image" then the plate definition covers the number of pixels comprising that image. It's a way of enumerating the number of steps/repitions to define the thing of interest

## 5.2 Training a Neural Causal Model

- Load & Prepare training data
- Evaluate model architecture
- Write a training procedure
- Implement tools for evaluating training progress
   

### 5.2.1 Setting up the Training Data

In [1]:
import torch

# love for our macbook users

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

USE_CUDA = True if device.type == 'cuda' else False

In [2]:
from torch.utils.data import Dataset

import numpy as np
import polars as pl
from torchvision import transforms

In [3]:
class CombinedDataset(Dataset):
    """This class loads and processes a dataset that combines MNIST and Typeface MNIST.
    Output is a torch.utils.data.Dataset object"""

    def __init__(self, csv_file):
        self.dataset = pl.read_csv(csv_file)
    
    def __len__(self):
        return len(self.dataset)
    
    def __getitem__(self, idx):
        
        # Load, normalize, and reshape to 28x28
        images = self.dataset.row(idx)[3:]
        images = np.array(images, dtype='float32')/255.
        images = images.reshape(28, 28)
        transform  = transforms.ToTensor()
        images = transform(images)

        # Get and process digit labels 0-9
        digits = self.dataset.row(idx)[2]
        digits = np.array([digits], dtype=int)

        # 1 for handwritten digits, 0 for typed digits
        is_handwritten = self.dataset.row(idx)[1]
        is_handwritten = np.array([is_handwritten], dtype='float32')

        # Tuple of image, digit label, and is_handwritten label
        return images, digits, is_handwritten

In [4]:
# using dataloader to load data and split into train and test

from torch.utils.data import DataLoader
from torch.utils.data import random_split
combined_mnist_url = "https://raw.githubusercontent.com/altdeep/causalML/master/datasets/combined_mnist_tmnist_data.csv"
def setup_dataloaders(batch_size=64):
    combined_dataset = CombinedDataset(combined_mnist_url)
    n = len(combined_dataset)
    
    # train on 80% of the data, test on 20%
    
    train_size = int(0.8 * n)
    test_size = n - train_size
    train_dataset, test_dataset = random_split(
        combined_dataset,
        [train_size, test_size],
        generator=torch.Generator().manual_seed(42)
    )

    # Create training and test loaders

    kwargs = {'num_workers': 0, 'pin_memory': True if device.type == 'cuda' else False}  # num_workers=0 avoids multiprocessing pickle issues in notebooks
    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        **kwargs
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=True,
        **kwargs
    )

    return train_loader, test_loader



### 5.2.2 Seetting up the variational autoencoder

Author demonstrates a latent variable "Z" as a compressed encoding of image information and presents it as a cause to the image itself in the DAG.

It's a stand-in for a set of latent causes, such as line thickness and font, rather than the direct representation of underlying causes it may represent. 

VAE trains 2 DNNs:

- Encoder: Encodes the image into a value for Z
- Decoder: Aligns with the DAG by generating an image from the "digit" label, is_handwriten label, and Z value

Decoder will use the latent variable Z + digit and is_handwritten to generate a 784 dimensional vector representing the 28x28 image

In [5]:
from torch import nn

class Decoder(nn.Module):
    """A class for the decoder used in the VAE"""

    def __init__(self, z_dim, hidden_dim) -> None:
        super().__init__()
        img_dim = 28 * 28

        digit_dim = 10
        is_handwritten_dim = 1

        # Softplus and sigmoid are nonlinear transforms (activation functions) used in mapping between layers
        self.softplus = nn.Softplus()
        self.sigmoid = nn.Sigmoid()

        encoding_dim = z_dim + digit_dim + is_handwritten_dim

        # fc1 is a linear function that maps the Z fector, the digit, and is_handwritten to a linear output
        # which is passed through a softplus activation function to create a hidden layer, a vector
        # whose length is given by hidden_layer

        self.fc1 = nn.Linear(encoding_dim, hidden_dim)
        
        # fc2 linearly maps the hidden layer to an output passed to a sigmoid function, resulting in a value between 1 and 0
        self.fc2 = nn.Linear(hidden_dim, img_dim)  

    def forward(self, z, digit, is_handwritten) -> torch.Tensor:
        """Define the forward computation from the latent Z variable to a generated X variable value"""

        input = torch.cat([z, digit, is_handwritten], dim=1)  # cobines Z and the labels
        hidden = self.softplus(self.fc1(input))  # compute the hidden layer
        
        # pass hidden layer to linear transform then to sigmoid transform to output a parameter vector length of 768. 
        # each element corresponsds to Bernoulli parameter value for an image pixel

        img_param = self.sigmoid(self.fc2(hidden))  

        return img_param

Causal DAG acts as a scaffold for causal probablistic ml model, with a decoder defining joint probability distribution in {is-handwritten, digit, X, Z}.

Bernouli is a hack because pixels are actually greyscale in this case, not just black or white. 

In [6]:
# Listing 5.5 - The causal model

import pyro
import pyro.distributions as dist

# Disabling distribution validation lets pyro calculate log likelihoods for pixels even though pixels are non-binary
dist.enable_validation(False)
def model(self, data_size=1):
    """The model of a single image. within the method, we register the decoder, a PyTorch module, with Pyro.
    This lets Pyro know about the parameters inside the decoder network"""

    pyro.module("decoder", self.decoder)
    options = dict(dtype=torch.float32, device=device)
    # We model the joint probabiility of Z, digit, and is_handwritten, sampling from each canonical distribution.
    # We sample Z from a multivariate normal with location paramater z_loc (all zeroes) and scale parameter z_scale (all ones)

    z_loc = torch.zeros(data_size, self.z_dim, **options)
    z_scale = torch.ones(data_size, self.z_dim, **options)

    z = pyro.sample("Z", dist.Normal(z_loc, z_scale).to_event(1))

    # We also sample the digit from a one-hot categorical distribution.
    # Equal probability is assigned to each digit

    p_digit = torch.ones(data_size, 10, **options) / 10

    digit = pyro.sample(
        "digit",
        dist.OneHotCategorical(p_digit)
    )

    # We similarly sample the is_handwritten variable from a bernoulli distribution
    p_is_handwritten = torch.ones(data_size, 1, **options) / 2
    is_handwritten = pyro.sample(
        "is_handwritten",
        dist.Bernoulli(p_is_handwritten).to_event(1)
    )


    # Decoder maps digit, is_handwritten, and Z to a probability parameter vector
    img_param = self.decoder(z, digit, is_handwritten)

    # Parameter vecotr is passed to Bernoulli distribution, modeling pixel values in the data (acknowledging the hack that 
    # the true pixel value sare not Bernoulli distributed)

    img = pyro.sample("img", dist.Bernoulli(img_param).to_event(1))

    return img, digit, is_handwritten

In [7]:
# listing 5.6 Applying model to N images in data

def training_model(self, img, digit, is_handwritten, batch_size):
    """The model represnts the DGP for one image. The training_model applies that model to the N images in the training data"""

    # Now we condition the model on evidence in the training data
    conditioned_on_data = pyro.condition(
        self.model,
        data={
            "digit": digit,
            "is_handwritten": is_handwritten,
            "img": img
        }
    )

    # This context manager rpresents the N-size plate representing repeat IID examples in the data in 
    # figure 5.9. In this case, N is the batch size. it works like a for loop, iterating over each data
    # unit in the batch.

    with pyro.plate("data", batch_size):  # I wasn't aware that plates are pyro native, cool
        img, digit, is_handwritten = conditioned_on_data(batch_size)

    return img, digit, is_handwritten

Model gives us joint distribution of {Z, X, digit, is_handwritten}

Sinze Z is latent, need to learn P(Z|X, digit, is_handwritten)

Distrribution of Z given X and labels is complex because we're using a decoder neural net to derive it. 

**Variational Inference** is a technique where we first define the approximating distribution

Q(Z|X, digit, is_handwritten) and try to make the distribution as close to P(Z|X, digit, is_handwritten) as possible

Encoder is the main ingredient in doing this, compressing information of the image into a lower dimensional encoding

In [8]:
# 5.7 Implement the Encoder

class Encoder(nn.Module):
    def __init__(self, z_dim, hidden_dim):
        super().__init__()
        img_dim = 28 * 28
        digit_dim = 10
        is_handwritten_dim = 1

        # In this encoder, we'll only use the softplus transform (activation function)
        self.softplus = nn.Softplus()

        # The linear transform fc1 combines with the softplus to map the 784-dimensional pixel vector, 10-dimensional
        # digit label vector, and 2-dimensional is_handwritten vector to the hidden layer

        input_dim = img_dim + digit_dim + is_handwritten_dim
        self.fc1 = nn.Linear(input_dim, hidden_dim)

        # The linear transforms, fc21 and fc22 will combine with the softplus to map the hidden vector to Z's vector space
        self.fc21 = nn.Linear(hidden_dim, z_dim)
        self.fc22 = nn.Linear(hidden_dim, z_dim)

    def forward(self, img, digit, is_handwritten):
        """Define the reverse computation from an observed X variable value to a latent Z variable value"""

        # combine image vector, digit label, and is_handwritten label int one input
        input = torch.cat([img, digit, is_handwritten], dim=1)
        
        # Map the input to the hidden layer
        hidden = self.softplus(self.fc1(input))

        # the VAE framework will sample Z from a normal distribution that approximates P(Z|img, digit, is_handwritten)
        # the final transforms map the hidden layer to a location and scale parameter for that normal distribution
        z_loc = self.fc21(hidden)
        z_scale = torch.exp(self.fc22(hidden))
        
        return z_loc, z_scale


The encoder output produces parameters of a distrution on Z. Guide function below uses encoder to sample values of Z

In [9]:
# Listing 5.8, the guide function

def training_guide(self, img, digit, is_handwritten, batch_size):
    """Method of the VAE that will use the encoder"""

    # Register the encoder so Pyro is aware of its weight parameters
    options = dict(dtype=torch.float32, device=device)

    # same plate context manaager for iterating over batch data that we used in training_model
    with pyro.plate("data", batch_size):
        
        # Use the encoder to map an image and its labels to parameters of a normal distribution
        z_loc, z_scale = self.encoder(img, digit, is_handwritten)
        normal_dist = dist.Normal(z_loc, z_scale).to_event(1)
        
        # Sample Z from the normal distribution
        z = pyro.sample("Z", normal_dist)

Steps:

 - Initialize latent dimension of Z to be 50
 - Hidden dimension to 400 in both encoder and decoder
 - Given 28 x28  image, 1d "is_handwritten" and 10d "digit", 28*28+1+10 = 795d feature vector
 - compress to 400d hidden layer
 - compress that to 50d location and scale parameter for Z's multivariate normal distribution

Architectural decisions above are determined by experimentation with the problem at hand

In [10]:
# Listing 5.9 - Full VAE Class

class VAE(nn.Module):
        def __init__(
            self,
            z_dim=50,  # latent dimensions == 50
            hidden_dim=400,  # hidden layers dimension == 400
            use_cuda=USE_CUDA
            ):
                super().__init__()
                self.use_cuda = use_cuda
                self.z_dim = z_dim
                self.hidden_dim = hidden_dim
                self.setup_networks()
        
        def setup_networks(self):
                """Set up the encoder and decoder"""

                self.encoder = Encoder(self.z_dim, self.hidden_dim)
                self.decoder = Decoder(self.z_dim, self.hidden_dim)

                # Move model to the appropriate device (CUDA, MPS, or CPU)
                self.to(device)

        model = model
        training_model = training_model
        training_guide = training_guide


### 5.2.3 The Training Procedure

Measuring goodness of a model means minimizing "reconstruction error" - how closely a generated image resembles the encoded image

The "Variational Inference" algorithm is favored in the book because its scalability to image/video data and robustness to latent variables

"In practice, soely minimizing reconstruction error leads to overfitting" so opts for probabilistic approach instead.

Probabilistic approach uses guide function to maximize likelihood of Z wrt P(Z|image, is-handwritten, digit) and likelihood of original image wrt P(image|is-handwritten,digit,Z)

ELBO: expected lower bound on log-likelihood of data

Variational Distribution: Q(Z|X, is-handwritten, digit)

Variational inference is a training approach that optimizes the weights of the encoder towards maximum likelihood of Z (P(Z|image, is-hadnwritten, digit))



In [11]:
# Listing 5.10 Helper function for plotting images

import matplotlib.pyplot as plt

def plot_image(img, title=None):
    fig = plt.figure()
    plt.imshow(img.cpu(), cmap="Greys_r", interpolation="nearest")
    if title is not None:
        plt.title(title)
    
    plt.show()

In [12]:
# Listing 5.11 Define a helper function for reconstructing and viewing images

import torch.nn.functional as F

def reconstruct_img(vae, img, digit, is_hw, use_cuda=USE_CUDA):
    """Given an input image, reconstruct the image by passing it through the encoder and then through the decoder"""

    img = img.reshape(-1, 28 * 28)
    digit = F.one_hot(torch.tensor(digit), 10)
    is_hw = torch.tensor(is_hw).unsqueeze(0)

    # Move tensors to the correct device
    img = img.to(device)
    digit = digit.to(device)
    is_hw = is_hw.to(device)

    z_loc, z_scale = vae.encoder(img, digit, is_hw)
    z = dist.Normal(z_loc, z_scale).sample()

    img_expectation = vae.decoder(z, digit, is_hw)

    return img_expectation.squeeze().view(28, 28).detach()

def compare_images(img1, img2):
    """Plots the two images ide by side for comparison"""

    fig = plt.figure()

    ax0 = fig.add_subplot(121)
    plt.imshow(img1.cpu(), cmap="Greys_r", interpolation="nearest")
    plt.axis("off")
    plt.title("original")

    ax1 = fig.add_subplot(122)
    plt.imshow(img2.cpu(), cmap="Greys_r", interpolation="nearest")
    plt.axis("off")
    plt.title("reconstruction")

    plt.show()

In [13]:
# Listing 5.12 Data Processing helper functions for training

def get_random_example(loader):
    """Choose a random example from the dataset"""

    random_idx = np.random.randint(0, len(loader.dataset))
    img, digit, is_handwritten = loader.dataset[random_idx]

    return img.squeeze(), digit, is_handwritten

def reshape_data(img, digit, is_handwritten):
    """Reshape the data"""

    digit = F.one_hot(digit, 10).squeeze()
    img = img.reshape(-1, 28*28)
    return img, digit, is_handwritten

def generate_coded_data(vae, use_cuda=USE_CUDA):
    """Generate data that is encoded"""

    z_loc = torch.zeros(1, vae.z_dim)
    z_scale = torch.ones(1, vae.z_dim)
    z = dist.Normal(z_loc, z_scale).to_event(1).sample()

    p_digit = torch.ones(1, 10)/10
    digit = dist.OneHotCategorical(p_digit).sample()

    p_is_handwritten = torch.ones(1, 1)/2
    is_handwritten = dist.Bernoulli(p_is_handwritten).sample()

    # Move tensors to the correct device
    z = z.to(device)
    digit = digit.to(device)
    is_handwritten = is_handwritten.to(device)

    img = vae.decoder(z, digit, is_handwritten)
    return img, digit, is_handwritten

def generate_data(vae, use_cuda=USE_CUDA):
    """Generate unencoded data"""

    img, digit, is_handwritten = generate_coded_data(vae, use_cuda)
    img = img.squeeze().view(28, 28).detach()
    digit = torch.argmax(digit, 1)

    is_handwritten = torch.argmax(is_handwritten, 1)
    return img, digit, is_handwritten



In [14]:
# Listing 5.13 Set up the training procedure

from pyro.infer import SVI, Trace_ELBO
from pyro.optim import Adam

pyro.clear_param_store()  # clear any values of parameters in the guide memory
vae = VAE()  # initialize the VAE

train_loader, test_loader = setup_dataloaders(batch_size=256)  # load the data
svi_adam = Adam({"lr": 1.0e-3})  # initialize optimizer

model = vae.training_model
guide = vae.training_guide

# Initialize SVI loss calculator. Loss negative "expected lower bound" (ELBO)
svi = SVI(model, guide, svi_adam, loss=Trace_ELBO())

With generative models, set up a procedure that uses data to evaluate how well training is progressing

In this example, Robert calculates and pritns loss fucntion on test data, just to make sure loss is progressively decreasing

In [15]:
# Listing 5.14 Setting up a test evaluation procedure

def test_epoch(vae, test_loader):
    """Calculate and print test loss"""

    epoch_loss_test = 0
    for img, digit, is_hw in test_loader:
        batch_size = img.shape[0]
        # Move data to the correct device
        img = img.to(device)
        digit = digit.to(device)
        is_hw = is_hw.to(device)
        
        img, digit, is_hw = reshape_data(
            img, digit, is_hw
        )
        
        epoch_loss_test += svi.evaluate_loss(
            img, digit, is_hw, batch_size
        )

    test_size = len(test_loader.dataset)
    avg_loss = epoch_loss_test / test_size
    print(f"Epoch: {epoch} avg. test loss: {avg_loss}")
    print("Comparing a random teest image to its reconstruction.")

    # compare random test image to its reconstruction
    random_example = get_random_example(test_loader)

    img_r, digit_r, is_hw_r = random_example

    img_recon = reconstruct_img(vae, img_r, digit_r, is_hw_r)

    compare_images(img_r, img_recon)

    # Generate a random image from the model

    print("Generate a random image from the model")

    img_gen, digit_gen, is_hw_gen = generate_data(vae)
    plot_image(img_gen, "Generated Image")

    print(f"Intended digit: {int(digit_gen)}")
    print(f"Intended as handwritten: {bool(is_hw_gen ==1)}")

# Listing 5.15 running trianing and plotting progress

NUM_EPOCHS = 250
TEST_FREQUENCY = 10

train_loss = []
train_size = len(train_loader.dataset)

for epoch in range(0, NUM_EPOCHS+1):  # run the training procedure for certain number of epochs
    loss = 0
    print("running")
    for img, digit, is_handwritten in train_loader:

        print("evaluating train_loader")
        batch_size = img.shape[0]

        # Move data to the correct device
        img = img.to(device)
        digit = digit.to(device)
        is_handwritten = is_handwritten.to(device)

        img, digit, is_handwritten = reshape_data(
            img, digit, is_handwritten
        )

        # Run a training step on one batch in one epoch
        loss += svi.step(
            img, digit, is_handwritten, batch_size
        )

        avg_loss = loss / train_size

        print(f"Epoch: {epoch} avg training loss: {loss}")

        train_loss.append(avg_loss)

        # The test data evaluation procedure runs every 10 epochs
        if epoch % TEST_FREQUENCY == 0:
            test_epoch(vae, test_loader)

        

Though Z (the encoded version of the image) is a stand-in for causal factors, a full representation of the image isn't actually a good way to approach causality. AS we progress through the book, we'll develop an intuition for how to sus out CAUSAL INFORMATION and what the representation should look like. 

### 5.3.1 Independence of mechanism as an inductive bias

Where O = Outcome and C = Cause, book asks us to reason about P(C) given P(O|C)

Scenarios:
 - C = Studying, O = Test Scores
 - C = Sun Scren, O = Sunburn

If asked to reason the probability that someone used sun screen given that they are sunburned, we're being asked to model P(C|O)

**Inductive bias** are the assumptions that lead an inference algorithm to perfer certain inferences or predictions over others. 

Causal models provide inductive biases that can drive better results and performance than other types of inductive bias. Example is that knowing P(O) can give some signal. Knowing P(O) for test scores can tell us whether students may have been lax regarding their studies, is an example author cited.

### 5.3.2 Semi-supervised Learning Case study

Independence of mechanis m can help us determine when semi-supervised learning will be effective

Defines semi-supervised in a very simple way

 - some percent of the data pairs X (characteristics) with Y (outcomes)
 - some percent of the data is just X
 - reasoning over the distribution of X may help us better understand and provide insight to P(Y|X)
 

The task is learning joint distribution of X and Y - P(X, Y) and use to to predict P(Y|X)